# Feature scaling and encoding

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Why scale?
Suppose you have two features: age (20 to 80) and salary (1000 to 5000). In Euclidean distance, a 1000 difference in salary counts 1000 times more than one year of age, so **age is practically ignored**.

[[lab:scaling]]

## Which models need it?
| Need it | Do not need it |
|---|---|
| KNN, K-Means, DBSCAN (distances) | Decision trees |
| SVM, SVR | Random forests |
| Regularized regression (Ridge, Lasso) | Gradient boosting |
| Neural networks and anything trained by gradient descent | Naive Bayes |
| PCA | |

## Methods
| Method | Formula | Result | Good for |
|---|---|---|---|
| `StandardScaler` | $z=(x-\mu)/\sigma$ | Mean 0, standard deviation 1 | A good default |
| `MinMaxScaler` | $(x-\min)/(\max-\min)$ | Between 0 and 1 | Images, neural networks |
| `RobustScaler` | $(x-\text{median})/IQR$ | Robust to outliers | Data with outliers |
| Log transform | $\log(1+x)$ | Less skew | Income, prices, counts |

## A worked example
Ages: 20, 30, 40, 50, 60. Mean 40, standard deviation 14.14:

| Age | Standard | MinMax |
|---|---|---|
| 20 | −1.41 | 0.00 |
| 30 | −0.71 | 0.25 |
| 40 | 0.00 | 0.50 |
| 50 | 0.71 | 0.75 |
| 60 | 1.41 | 1.00 |



In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

age = np.array([[20], [30], [40], [50], [60], [95]])   # 95 is an outlier
for s in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    print(f"{type(s).__name__:15s}", s.fit_transform(age).ravel().round(2))



## Encoding categorical features
| Method | Example | When |
|---|---|---|
| One-hot | color → three 0/1 columns red/green/blue | Unordered categories |
| Ordinal | low=0, medium=1, high=2 | **Ordered** categories |
| Target encoding | Mean target per category | Very many categories (with care) |

> Giving 0, 1, 2 to "Tehran, Shiraz, Tabriz" tells the model that Shiraz lies between Tehran and Tabriz! Unordered categories need one-hot encoding.

## Data leakage
Fit the scaler **on the training data only**; otherwise information from the test data (its mean and standard deviation) leaks into training. `Pipeline` and `ColumnTransformer` do this correctly by themselves:



In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = pd.DataFrame({
    "age": [25, 32, 47, 51, 23, 36, 60, 29, 41, 55],
    "salary": [1800, 2500, 4100, 3900, 1500, 3000, 4500, 2100, 3300, 4200],
    "city": ["Tehran", "Shiraz", "Tehran", "Tabriz", "Shiraz", "Tabriz", "Tehran", "Shiraz", "Tabriz", "Tehran"],
    "level": ["low", "medium", "high", "high", "low", "medium", "high", "low", "medium", "high"],
    "bought": [0, 0, 1, 1, 0, 1, 1, 0, 1, 1],
})
X, y = df.drop(columns="bought"), df["bought"]
prep = ColumnTransformer([
    ("num", StandardScaler(), ["age", "salary"]),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["city"]),
    ("level", OrdinalEncoder(categories=[["low", "medium", "high"]]), ["level"]),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression())])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model.fit(X_tr, y_tr)                 # the scaler only sees the training rows
print(prep.get_feature_names_out())
print("Test accuracy:", model.score(X_te, y_te))